# PubMed Systematic Review Pipeline

*Author: Regina Chua*

> This is the single notebook I run for the PubMed arm of my systematic review. I built it to be fully reproducible so my collaborators (and future me) can re-run the exact same search and analysis without guessing at parameters. Anything that defines *what* I'm searching for lives in `search_strategy.py`, I kept out of this notebook on purpose so the query stays identical (or up-to-date as I revise it) across every database I touch.

This notebook consolidates the previous `pubmed_pull` and `pubmed_exploration` notebooks
(now in `archive/`) into a single workflow with two goals:

1. **Collect** the set of articles that meet the search-strategy criteria from PubMed.
2. **Analyse** the resulting corpus with NLP (TF-IDF + n-gram ranking) to surface the key terms.

All inclusion/exclusion criteria live in `search_strategy.py` so the query stays consistent
across databases. Run the cells top-to-bottom.

**References:** PubMed access via [`pymed`](https://github.com/gijswobben/pymed).


## 1. Environment Setup

> Here I just pull in every library the pipeline needs and load my secrets from `.env`. My goal is to keep my PubMed contact email *out* of the code. PubMed asks for an email with each request (though it's not required at this time), and I don't want it committed to git so I read it from the environment instead. The print line is a quick sanity check that the email actually loaded before I start querying.


In [21]:
from datetime import datetime
from pathlib import Path
import json
import os
import re

from dotenv import load_dotenv
import numpy as np
import pandas as pd
from pymed import PubMed
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    _HAS_PLOTLY = True
except Exception:
    _HAS_PLOTLY = False

# Load PUBMED_EMAIL (and any other secrets) from .env
load_dotenv()
pd.set_option("display.max_colwidth", 120)
print("Environment loaded. PUBMED_EMAIL set:", bool(os.getenv("PUBMED_EMAIL")))

Environment loaded. PUBMED_EMAIL set: False


## 2. Search Strategy

>This is the heart of the review and the part I'm most careful about. Rather than hard-coding a query string (as I did in the exploration script), I import my inclusion/exclusion terms from `search_strategy.py` and build the boolean query from them. My goal is to have **one single source of truth**: when I (or my supervisors) refine the terms, I only edit `search_strategy.py` and every database notebook stays in sync. The structure is essentially `(disease) AND (spatial) AND (exposure) NOT (exclusions)`, which mirrors my review scope: a neurodegenerative disease, a spatial/geographic angle, and an environmental exposure.

In [22]:
from search_strategy import (
    INCLUSION_CRITERIA,
    EXCLUSION_TERMS,
    DATE_FILTER,
    CLEANING_RULES,
    ALTERNATE_TERMS
)


def or_group(terms):
    """Quote each term and join with OR inside parentheses."""
    return "(" + " OR ".join(f'\"{t}\"' for t in terms) + ")"


def build_pubmed_query(inclusion, exclusion):
    """Compose the PubMed boolean query from the centralized search strategy.

    Structure: (disease) AND (spatial) AND (exposure) NOT (exclusions)
    """
    include = " AND ".join([
        or_group(inclusion["disease"]),
        or_group(inclusion["spatial"]),
        or_group(inclusion["exposure"]),
    ])
    return f"{include} NOT {or_group(exclusion)}"


def merge_terms(primary, alternates):
    """Combine primary inclusion terms with alternate/NLP-suggested terms per
    category (order-preserving, case-insensitive de-duplication)."""
    merged = {}
    for category, base_terms in primary.items():
        seen, combined = set(), []
        for t in list(base_terms) + list(alternates.get(category, [])):
            if t.lower() not in seen:
                seen.add(t.lower())
                combined.append(t)
        merged[category] = combined
    return merged

print("Search criteria initialized.")


Search criteria initialized.


Change the line below to `True` to include the alternate terms. `False` otherwise.

In [23]:
# Fold the NLP-suggested ALTERNATE_TERMS into the inclusion criteria so the
# query reflects the expanded vocabulary. Set this to False to fall back to the
# core criteria only (useful for comparing search sensitivity).
INCLUDE_ALTERNATE_TERMS = True
search_criteria = (
    merge_terms(INCLUSION_CRITERIA, ALTERNATE_TERMS)
    if INCLUDE_ALTERNATE_TERMS
    else INCLUSION_CRITERIA
)

query = build_pubmed_query(search_criteria, EXCLUSION_TERMS)
start_date = DATE_FILTER["start_date"]
end_date = DATE_FILTER["end_date"]

print("Date window:", start_date, "to", end_date)
print("Alternate terms folded into query:", INCLUDE_ALTERNATE_TERMS)
for category, term_list in search_criteria.items():
    print(f"{category} terms ({len(term_list)}):", term_list)
print("\nQuery:\n", query)

Date window: 2020-01-01 to 2025-12-31
Alternate terms folded into query: True
disease terms (14): ['parkinson*', 'parkinson* disease', 'MSA', 'multiple system atrophy', 'DLB', 'dementia with lewy bodies', 'PSP', 'progressive supranuclear palsy', 'CBS', 'corticobasal syndrome', 'CBD', 'corticobasal degeneration', 'synucleinopath*', 'synuclein*']
spatial terms (24): ['geospatial*', 'geograph*', 'environment*', 'spatiotemporal', 'spatial*', 'GIS', 'geographic information systems', 'spatial interpolation', 'spatial epidemiology', 'remote sens*', 'latitude', 'longitude', 'clustering', 'residence', 'administrative division', 'drone', 'imagery', 'landsat', 'map', 'mapping', 'modis', 'satellite', 'sentinel', 'topologic*']
exposure terms (25): ['pollut*', 'chemical', 'pesticide*', 'air pollution', 'microplastic pollution', 'traffic pollution', 'water pollution', 'trichloroethylene', 'air quality', 'exposure', 'environment*', 'particulate*', 'atmospher*', 'carbon', 'humidity', 'meteorologic*', '

## 3. Helper Functions

>These are the small utilities I lean on throughout the notebook. I define them once, up front, so the rest of the workflow stays readable (and to avoid the bug I had in the old notebook where a helper was trapped inside an `if` block - ah well, live and learn). `article_to_flat_dict` flattens the messy `pymed` objects into plain rows, and `row_to_text` / `normalize_term` give me clean, combined text per article for the NLP step. My goal here is to handle missing fields, list-valued keywords, and odd data types without crashing the run.


In [12]:
def is_empty(obj):
    """Safe emptiness check for lists, Series, numpy/sparse arrays."""
    if obj is None:
        return True
    try:
        return len(obj) == 0
    except Exception:
        pass
    try:
        return getattr(obj, "size", 0) == 0
    except Exception:
        return False


def article_to_flat_dict(article):
    """Flatten a pymed article object into a plain dict."""
    try:
        raw = article.toJSON()
        if isinstance(raw, str):
            return json.loads(raw)
        if isinstance(raw, dict):
            return raw
    except Exception:
        pass
    return {
        "pubmed_id": getattr(article, "pubmed_id", None),
        "title": getattr(article, "title", None),
        "publication_date": str(getattr(article, "publication_date", "") or ""),
        "keywords": getattr(article, "keywords", None),
        "abstract": getattr(article, "abstract", None),
        "doi": getattr(article, "doi", None),
    }


def _coerce_text(val):
    """Normalize a single field value to a string (handles NA / list / ndarray)."""
    if val is None:
        return ""
    if isinstance(val, float) and np.isnan(val):
        return ""
    if isinstance(val, np.ndarray):
        return " ".join(map(str, val.tolist())) if val.size else ""
    if isinstance(val, (list, tuple)):
        return " ".join(map(str, val)) if len(val) else ""
    return str(val)


def row_to_text(row, cols=("title", "abstract", "keywords")):
    """Combine the chosen text columns of a row into one string."""
    get = row.get if isinstance(row, dict) else (lambda c, d="": row[c] if c in row else d)
    parts = [_coerce_text(get(c, "")) for c in cols]
    return " ".join(p for p in parts if p)


def normalize_term(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    return re.sub(r"\s+", " ", s).strip()


## 4. Collect Articles from PubMed

>This is the actual data collection step. I send my query to PubMed, flatten the results into a DataFrame, and then restrict to the date window from `search_strategy.py` (currently a 5-year window). I keep `results` as a list so I can re-use it without re-hitting the API. The two printed counts (total returned vs. within the date window) are my first checkpoint that the search behaved as expected.


In [13]:
# PubMed kindly requests a tool name and contact email
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="SystematicReview", email=os.getenv("PUBMED_EMAIL"))

MAX_RESULTS = 500

# Convert to a list so results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=MAX_RESULTS))
df_results = pd.json_normalize([article_to_flat_dict(a) for a in results])

# Restrict to the configured date window
df_results["publication_date"] = pd.to_datetime(
    df_results["publication_date"], format="ISO8601", errors="coerce"
)
mask = df_results["publication_date"] >= start_date
if end_date:
    mask &= df_results["publication_date"] <= end_date
df_filtered = df_results[mask].copy()

preview_cols = [c for c in ["title", "abstract", "publication_date", "keywords"]
                if c in df_results.columns]

print("Total articles returned:", df_results.shape[0])
print("Articles within date window:", df_filtered.shape[0])
display(df_filtered[preview_cols].head())

Total articles returned: 500
Articles within date window: 309


,title,abstract,publication_date,keywords
190,Agricultural Copper Pesticide Exposure and Metabolic Profiles among Parkinson's Disease Cases and Community Controls...,Copper-based pesticide exposures have been linked to increased Parkinson's disease (PD) risk through oxidative stres...,2025-12-29,[]
191,"Short-term associations between ambient air pollution and emergency department visits for Parkinson disease, multipl...","Previous studies have linked ambient air pollution to worsening neurological conditions, but research in the United ...",2025-12-26,"[Air pollution, Emergency department visits, Health effect, Migraine, Multiple sclerosis, Parkinson disease, Seizure]"
192,"Large differences in MMR and DTaP-IPV vaccination coverage among primary schools by denomination, the Netherlands, 2...","Recently, the number of measles and pertussis cases increased worldwide, including in the Netherlands. As schools ar...",2025-12-25,"[Primary schools, Vaccination coverage, Vaccine-preventable diseases]"
193,Navigation in Virtual Reality Floor Mazes: Added Cognitive Demand and Its Effects on Gait and Balance in Parkinson's...,"Along with motor dysfunction, people with Parkinson's Disease (PD) often develop cognitive dysfunction, linked to th...",2025-12-24,[]
194,Optimized Selective Media Enhance the Isolation and Characterization of Gut-Derived Probiotic Yeasts.,This study applied a guided culturomics workflow to isolate and characterize gut-associated yeasts as probiotic cand...,2025-12-24,"[gut fungi, mycobiome, probiotic yeast, yeast]"


## 5. Clean Results

>Time to tidy the raw pull. I drop duplicate titles, and I require both a PubMed ID and a DOI so every article is uniquely identifiable and traceable for screening. I drive these rules from `CLEANING_RULES` in `search_strategy.py`.

In [14]:
if CLEANING_RULES.get("remove_duplicate_titles", True):
    df_filtered = df_filtered.drop_duplicates(subset=["title"])
if CLEANING_RULES.get("require_pubmed_id", True) and "pubmed_id" in df_filtered.columns:
    df_filtered = df_filtered.dropna(subset=["pubmed_id"])
if CLEANING_RULES.get("require_doi", True) and "doi" in df_filtered.columns:
    df_filtered = df_filtered.dropna(subset=["doi"])

print("Articles after cleaning:", df_filtered.shape[0])
display(df_filtered[preview_cols].head())

Articles after cleaning: 307


,title,abstract,publication_date,keywords
190,Agricultural Copper Pesticide Exposure and Metabolic Profiles among Parkinson's Disease Cases and Community Controls...,Copper-based pesticide exposures have been linked to increased Parkinson's disease (PD) risk through oxidative stres...,2025-12-29,[]
191,"Short-term associations between ambient air pollution and emergency department visits for Parkinson disease, multipl...","Previous studies have linked ambient air pollution to worsening neurological conditions, but research in the United ...",2025-12-26,"[Air pollution, Emergency department visits, Health effect, Migraine, Multiple sclerosis, Parkinson disease, Seizure]"
192,"Large differences in MMR and DTaP-IPV vaccination coverage among primary schools by denomination, the Netherlands, 2...","Recently, the number of measles and pertussis cases increased worldwide, including in the Netherlands. As schools ar...",2025-12-25,"[Primary schools, Vaccination coverage, Vaccine-preventable diseases]"
193,Navigation in Virtual Reality Floor Mazes: Added Cognitive Demand and Its Effects on Gait and Balance in Parkinson's...,"Along with motor dysfunction, people with Parkinson's Disease (PD) often develop cognitive dysfunction, linked to th...",2025-12-24,[]
194,Optimized Selective Media Enhance the Isolation and Characterization of Gut-Derived Probiotic Yeasts.,This study applied a guided culturomics workflow to isolate and characterize gut-associated yeasts as probiotic cand...,2025-12-24,"[gut fungi, mycobiome, probiotic yeast, yeast]"


## 6. Export Cleaned Results

>I save the cleaned set to CSV so it becomes the stable hand-off artifact for the screening stage and for merging with the other database results later. Exporting here means I can pick the analysis back up from the CSV without re-querying PubMed.

In [ ]:
output_csv_path = Path("pubmed_results_cleaned_2026.csv")
df_filtered.to_csv(output_csv_path, index=False)
print(f"Exported {len(df_filtered)} cleaned articles to {output_csv_path.resolve()}")

Exported 411 cleaned articles to /mnt/c/Users/ReginaChua/Desktop/sysrev/pubmed_results_cleaned_2026.csv
